In [33]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [36]:
pre_dataset = pd.read_csv("h1b_kaggle.csv")
print(pre_dataset.count()) # get the count of records

Unnamed: 0            1688905
CASE_STATUS           1688905
EMPLOYER_NAME         1688876
SOC_NAME              1685486
JOB_TITLE             1688891
FULL_TIME_POSITION    1688904
PREVAILING_WAGE       1688887
YEAR                  1688905
WORKSITE              1688905
lon                   1637891
lat                   1637890
dtype: int64


In [37]:
pre_dataset.isnull().sum()

,0
Unnamed: 0,0
CASE_STATUS,0
EMPLOYER_NAME,29
SOC_NAME,3419
JOB_TITLE,14
FULL_TIME_POSITION,1
PREVAILING_WAGE,18
YEAR,0
WORKSITE,0
lon,51014


In [38]:
# preprocessing dataset
dataset = pre_dataset.drop(pre_dataset.columns[[9, 10]], axis=1).dropna()  # drop lat and lon and NA values
dataset.rename(columns={'Unnamed: 0':'ID'}, inplace=True)
print(dataset.count())

ID                    1685435
CASE_STATUS           1685435
EMPLOYER_NAME         1685435
SOC_NAME              1685435
JOB_TITLE             1685435
FULL_TIME_POSITION    1685435
PREVAILING_WAGE       1685435
YEAR                  1685435
WORKSITE              1685435
dtype: int64


In [39]:
print(f'Unique Employer Names:{len(dataset.EMPLOYER_NAME.unique())}')
print(f'Unique Job Titles:{len(dataset.JOB_TITLE.unique())}')

Unique Employer Names:140373
Unique Job Titles:177591


In [40]:
print(dataset.CASE_STATUS.unique())

['CERTIFIED-WITHDRAWN' 'WITHDRAWN' 'CERTIFIED' 'DENIED' 'REJECTED']


In [41]:
print(dataset['CASE_STATUS'].value_counts())

CASE_STATUS
CERTIFIED              1479583
CERTIFIED-WITHDRAWN     122409
WITHDRAWN                54539
DENIED                   28903
REJECTED                     1
Name: count, dtype: int64


In [42]:
# delete all outlier values in CASE_STATUS column
to_drop = ["INVALIDATED","REJECTED","PENDING QUALITY AND COMPLIANCE REVIEW - UNASSIGNED"]
final_dataset = dataset[~dataset['CASE_STATUS'].isin(to_drop)]
print(final_dataset['CASE_STATUS'].value_counts())

CASE_STATUS
CERTIFIED              1479583
CERTIFIED-WITHDRAWN     122409
WITHDRAWN                54539
DENIED                   28903
Name: count, dtype: int64


In [43]:
print(final_dataset.count())

ID                    1685434
CASE_STATUS           1685434
EMPLOYER_NAME         1685434
SOC_NAME              1685434
JOB_TITLE             1685434
FULL_TIME_POSITION    1685434
PREVAILING_WAGE       1685434
YEAR                  1685434
WORKSITE              1685434
dtype: int64


In [44]:
# Seperate independant and dependant variables
X = final_dataset.iloc[:,2:8].values
y = final_dataset.iloc[:,1].values

In [45]:
X[1]

array(['GOODMAN NETWORKS, INC.', 'CHIEF EXECUTIVES',
       'CHIEF OPERATING OFFICER', 'Y', 242674.0, 2016], dtype=object)

In [46]:
y[1]

'CERTIFIED-WITHDRAWN'

In [47]:
# encode categorical data
from sklearn.preprocessing import LabelEncoder

labelencoder_X = LabelEncoder()
X[:, 0] = labelencoder_X.fit_transform(X[:, 0])


labelencoder_X1 = LabelEncoder()
X[:, 1] = labelencoder_X1.fit_transform(X[:, 1])

labelencoder_X2 = LabelEncoder()
X[:, 2] = labelencoder_X2.fit_transform(X[:, 2])

labelencoder_X3 = LabelEncoder()
X[:, 3] = labelencoder_X3.fit_transform(X[:, 3])

labelencoder_X5 = LabelEncoder()
X[:, 5] = labelencoder_X5.fit_transform(X[:, 5])

In [48]:
X[1]

array([51204, 229, 26588, 1, 242674.0, 2], dtype=object)

In [49]:
# label the outcome
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)

In [50]:
y[1]

np.int64(1)

In [51]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [52]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [53]:
X_train[1]

array([-2.49334229e-01, -9.18463921e-01,  7.93884569e-01,  5.37689043e-01,
       -7.91895401e-04, -1.73374748e-01])

In [54]:
# Fitting Random Forest Classification to the Training set
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'gini', random_state = 0,n_jobs=-1)
classifier.fit(X_train, y_train)

RandomForestClassifier(n_estimators=10, n_jobs=-1, random_state=0)

In [55]:
# Predicting the Test set results
y_pred = classifier.predict(X_test)

In [56]:
# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[357012   5587   2798   4407]
 [ 21693   8374    102    345]
 [  6215    106    811    140]
 [ 11474    503    165   1627]]


In [57]:
# Calculate accuracy
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, y_pred)
print(acc)

0.8729468220685923
